In [1]:
import pandas as pd
import matplotlib.pyplot as plt


In [2]:
df1 = pd.read_csv("central_province.csv")
df2 = pd.read_csv("north_central_province.csv")
df3 = pd.read_csv("southern_province.csv")
df4 = pd.read_csv("uva_province.csv")
df5 = pd.read_csv("western_province.csv")
df6 = pd.read_csv("eastern_province.csv")
df7 = pd.read_csv("north_western_province.csv")
df8 = pd.read_csv("northern_province.csv")
df9 = pd.read_csv("sabaragamuwa_province.csv")

merged_df = pd.concat([df1, df2,df3,df4,df5,df6,df7,df8,df9], ignore_index=True)

merged_df
merged_df.to_csv("combined.csv", index=False)

In [3]:
import pandas as pd
import requests
import time
import random
from tqdm import tqdm

# ========================= CONFIG =========================
START_LAT = 7.1808
START_LON = 79.8841
BATCH_SIZE = 5          # You can increase to 10-20 if it becomes stable
MAX_RETRIES = 3
BASE_SLEEP = 1.8        # seconds between requests

# OSRM public demo server (unreliable for large jobs)
OSRM_URL = "http://router.project-osrm.org/table/v1/driving/"

# ========================================================

def get_real_distances_from_start(batch_df, lat_col='Latitude', lon_col='Longitude'):
    batch_df = batch_df.dropna(subset=[lat_col, lon_col]).copy()
    
    # Filter to Sri Lanka bounding box (very important for speed & accuracy)
    batch_df = batch_df[
        (batch_df[lat_col] >= 5.0) & (batch_df[lat_col] <= 10.0) &
        (batch_df[lon_col] >= 79.0) & (batch_df[lon_col] <= 82.0)
    ].copy()
    
    if len(batch_df) == 0:
        batch_df['real_distance_from_start'] = None
        return batch_df
    
    # Build coordinates: start point first, then all destinations
    coords = [(START_LON, START_LAT)] + list(zip(batch_df[lon_col], batch_df[lat_col]))
    coords_str = ";".join([f"{lon:.6f},{lat:.6f}" for lon, lat in coords])
    
    # Improved URL - explicitly request distance annotation
    url = f"{OSRM_URL}{coords_str}?sources=0&annotations=distance"
    
    try:
        response = requests.get(url, timeout=15)
        
        if response.status_code != 200:
            print(f"  HTTP Error {response.status_code}: {response.text[:300]}")
            batch_df['real_distance_from_start'] = None
            return batch_df
        
        data = response.json()
        
        # Debug info
        code = data.get("code", "Unknown")
        if code != "Ok":
            print(f"  OSRM Error: {code} - {data.get('message', '')}")
            batch_df['real_distance_from_start'] = None
            return batch_df
        
        # Extract distances (in meters) from the first row (source=0), skip the first value (self-distance)
        distances = data.get('distances', [[]])[0][1:]
        
        if len(distances) != len(batch_df):
            print(f"  Warning: Distance count mismatch ({len(distances)} vs {len(batch_df)})")
        
        # Convert to km
        batch_df['real_distance_from_start'] = [
            round(d / 1000.0, 3) if isinstance(d, (int, float)) and d is not None else None 
            for d in distances
        ]
        
        return batch_df
        
    except requests.exceptions.Timeout:
        print("  Request timed out")
    except requests.exceptions.ConnectionError:
        print("  Connection error")
    except Exception as e:
        print(f"  Unexpected error: {e}")
    
    batch_df['real_distance_from_start'] = None
    return batch_df


def get_real_distances_batched(df, batch_size=BATCH_SIZE):
    results = []
    
    print(f"Starting processing of {len(df)} points in batches of {batch_size}...\n")
    
    for i in tqdm(range(0, len(df), batch_size), desc="Processing batches"):
        batch = df.iloc[i:i + batch_size].copy()
        
        for attempt in range(MAX_RETRIES):
            processed_batch = get_real_distances_from_start(batch)
            
            # Success if at least some distances were calculated
            if processed_batch['real_distance_from_start'].notna().any():
                results.append(processed_batch)
                break
            else:
                if attempt < MAX_RETRIES - 1:
                    sleep_time = BASE_SLEEP + random.uniform(0.5, 2.0)
                    time.sleep(sleep_time)
                else:
                    # Final failure
                    processed_batch['real_distance_from_start'] = None
                    results.append(processed_batch)
        
        # Be nice to the public server
        time.sleep(BASE_SLEEP + random.uniform(0.3, 1.2))
    
    final_df = pd.concat(results, ignore_index=True)
    return final_df


# ========================= MAIN =========================
if __name__ == "__main__":
    df = pd.read_csv("combined.csv")
    
    print(f"Loaded {len(df)} rows from combined.csv")
    
    df = get_real_distances_batched(df, batch_size=BATCH_SIZE)
    
    output_file = "combined_with_real_distances.csv"
    df.to_csv(output_file, index=False)
    
    print("\n✅ Done!")
    print(f"Output saved to: {output_file}")
    print(f"Success rate: {df['real_distance_from_start'].notna().mean():.1%} "
          f"({df['real_distance_from_start'].notna().sum()}/{len(df)} points)")
    
    print("\nPreview:")
    print(df.head())

Loaded 736 rows from combined.csv
Starting processing of 736 points in batches of 5...



Processing batches: 100%|██████████| 148/148 [24:33<00:00,  9.96s/it]


✅ Done!
Output saved to: combined_with_real_distances.csv
Success rate: 100.0% (727/727 points)

Preview:
   Place_ID                                 Place_Name        City Province  \
0         1  Sri Dalada Maligawa (Temple of the Tooth)       Kandy  Central   
1         2         Royal Botanical Gardens Peradeniya  Peradeniya  Central   
2         3                       Kandy Lake Promenade       Kandy  Central   
3         4                Bahirawakanda Buddha Statue       Kandy  Central   
4         5                Udawattakele Forest Reserve       Kandy  Central   

    Category  Latitude  Longitude Terrain_Type  Time_Needed (H)  Rating  \
0  Religious    7.2936    80.6413        Paved              2.0     4.9   
1     Nature    7.2716    80.5966        Hilly              3.0     4.8   
2      Relax    7.2905    80.6409        Paved              1.5     4.6   
3  Religious    7.2945    80.6260        Hilly              1.5     4.7   
4     Nature    7.2974    80.6384     Off-R